<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">چهار بایت به‌ازای وزن، کل حافظه نیست</h1>
<p style="text-align:right">درس 88 از 92 · مدل بزرگ چه هزینه‌های تازه‌ای دارد؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">63-scale</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-01/63-scale.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">تخمین حافظهٔ وزن را از تخمین سادهٔ آموزش و مصرف واقعی <bdi dir="ltr">Tensor</bdi>ها جدا کنید.</p><p style="text-align:right">پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">numel</code>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">dtype</code>، <bdi dir="ltr">Gradient</bdi> و دو آرایهٔ میانگین متحرک <bdi dir="ltr">Adam</bdi>.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۳۰–۵۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">یک میلیون وزن <bdi dir="ltr">float32</bdi> تقریباً چهار میلیون بایت است؛ چرا آموزش به بیشتر از همین مقدار نیاز دارد؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,4,8,2,1,0.0))
optimizer = torch.optim.AdamW(model.parameters())
model(torch.tensor([[1,2,3]]),torch.tensor([[2,3,4]]))[1].backward()
optimizer.step()
parameter_count = sum(p.numel() for p in model.parameters())
optimizer_bytes = sum(value.numel()*value.element_size() for state in optimizer.state.values()
                      for value in state.values() if isinstance(value,torch.Tensor))
print('parameters:',parameter_count,'actual optimizer tensor bytes:',optimizer_bytes)
print('Activation memory and allocator overhead are not measured here.')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">memory_estimate(parameters, bytes_per_number, training)</code> یک تعداد بایت صحیح بدهد. برای وزن‌ها یک مجموعه و برای تخمین سادهٔ آموزش چهار مجموعهٔ هم‌اندازه در نظر بگیرید: وزن، <bdi dir="ltr">Gradient</bdi> و دو آرایهٔ میانگین متحرک <bdi dir="ltr">Adam</bdi>. این مدل تخمینی فرض می‌کند <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">dtype</code> هر چهار یکسان است.</p>
</div>

In [ ]:
def memory_estimate(parameters, bytes_per_number, training):
    # TODO: تخمین خام، نه حافظهٔ اوج واقعی
    return None

In [ ]:
def test_exercise():
    result = memory_estimate(1_000_000,4,False)
    if result is None:
        return False
    assert result==4_000_000
    assert memory_estimate(1_000_000,4,True)==16_000_000
    assert memory_estimate(100_000_000,4,False)==400_000_000
    assert memory_estimate(10,2,True)==80
    assert memory_estimate(0,4,True)==0
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: memory_estimate')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط تعداد بایت هر وزن را در تخمین تغییر دهید. این محاسبه مدل را <bdi dir="ltr">Quantize</bdi> نمی‌کند و سرعت اجرا را هم اندازه نمی‌گیرد.</p>
</div>

In [ ]:
for bytes_per_weight in (4,2,1):
    print(bytes_per_weight,parameter_count*bytes_per_weight,'estimated raw weight bytes')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">len(list(model.parameters()))</code> تعداد شیءهای <bdi dir="ltr">Parameter</bdi> را می‌شمارد، نه تعداد عددهای داخل آن‌ها را. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">model_weight_bytes(model)</code> مجموع حاصل‌ضرب <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">numel()</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">element_size()</code> همهٔ <bdi dir="ltr">Parameter</bdi>ها را بدهد؛ <bdi dir="ltr">Buffer</bdi>ها و <bdi dir="ltr">Activation</bdi>ها جزو این تابع نیستند.</p>
</div>

In [ ]:
wrong = len(list(model.parameters()))*4
print('wrong bytes from parameter-object count:',wrong)
print('first weight shape:',tuple(next(model.parameters()).shape))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def model_weight_bytes(model):
    # TODO: اندازهٔ هر Tensor و dtype واقعی آن
    return None

In [ ]:
def test_repair():
    result = model_weight_bytes(model)
    if result is None:
        return False
    assert result==parameter_count*4
    layer = torch.nn.Linear(3,2,bias=True).double()
    assert model_weight_bytes(layer)==(3*2+2)*8
    assert model_weight_bytes(torch.nn.Identity())==0
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: model_weight_bytes')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">وزن‌ها و وضعیت <bdi dir="ltr">Optimizer</bdi> از مدل واقعی‌اند؛ برآورد چهارمجموعه‌ای فقط یک مدل ساده است. <bdi dir="ltr">Mixed precision</bdi>، آموزش توزیع‌شده و <bdi dir="ltr">FlashAttention</bdi> در این دفتر اجرا نمی‌شوند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام هزینه‌ها هنوز در تخمین شما نیستند و چرا کم‌شدن بایت هر وزن الزاماً سرعت را بیشتر نمی‌کند؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-01/63-scale.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/63-scale.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>